## Transform Results Data
- Read bronze Results table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(constructorId->constructor_id,driverId->driver_id)
- Rename columns to make them more meaningful(grid->grid_position)
- Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)
- Remove duplicate records.
- Transform values of column race_name to Title Case
- Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
import pyspark.sql.functions as F

### Step 1 Read bronze results table

In [0]:
results_df=spark.table(bronze_table)


### Srep 2 -Keep only the columns required for analytics(Drop url column)

In [0]:
#flexible   `

results_selected_df=(
    results_df.drop("url")
)

### Step 3 &4 - Stamdardise column name
- Standardise column name using snakecase(constructorId->constructor_id,driverId->driver_id)


In [0]:
results_renamed_df=(
    results_selected_df.withColumnsRenamed(
        {"constructorId":"constructor_id",
         "driverId":"driver_id",
         "raceName":"race_name",
         "date":"race_date",
         "time":"race_time",
         "grid":"grid_position",
         "laps":"completed_laps",
         "number":"car_number",
         "position":"final_position",
         "positionText":"final_position_text"
        }
    )
)

#display(results_renamed_df)

### Step 5 Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)

In [0]:
results_valid_df=(
    results_renamed_df.filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("driver_id").isNotNull() &
        F.col("constructor_id").isNotNull()
    )
)



In [0]:
display(results_renamed_df.count()-results_valid_df.count())

11

### Step -6 Remove duplicates

In [0]:
#dropduplicates - you can give column name and distinct will be applied on that particular column only
results_distinct_df=results_valid_df.dropDuplicates(["season","round","driver_id","constructor_id"])
#display(results_distinct_df)



### Step -7 Transform values of column race_name to Title Case

In [0]:
results_final_df=(
    results_distinct_df
    .withColumn("race_name",F.initcap(F.col("race_name")))
)
#display(drivers_final_df)

### Step-7 Write the transformed data to silver drivers table

In [0]:
(
   results_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .option('overwriteSchema', 'true')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.read.table(silver_table))

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file
1950-05-13,British Grand Prix,1,1950,maserati,bira,5,49,21,0.0,14,R,Out of fuel,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,lago,cabantous,6,68,14,3.0,4,4,+2 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,maserati,chiron,11,26,19,0.0,18,R,Clutch,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,lago,claes,21,64,18,0.0,11,11,+6 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,alta,crossley,17,43,24,0.0,16,R,Transmission,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,lago,etancelin,14,65,16,0.0,8,8,+5 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,alfa,fagioli,2,70,3,6.0,2,2,Finished,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,alfa,farina,1,70,2,9.0,1,1,Finished,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,era,gerard,13,67,12,0.0,6,6,+3 Laps,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950-05-13,British Grand Prix,1,1950,maserati,graffenried,8,36,20,0.0,17,R,Engine,2026-10-06T17:48:53.135469Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
